In [ ]:
import sys
sys.path.append("../")

import numpy as np
import torch
import torch.nn as nn
np.random.seed(42)
torch.manual_seed(42)

from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.ar_model import AutoReg
from data_loader import PairData

data = PairData("../../data/real_pair_MCD_YUM.csv")
n  = len(data)
q1 = n // 4
q4 = 3 * n // 4

jres      = coint_johansen(np.log(data.s[:, :q1]).T, det_order=0, k_ar_diff=1)
cv        = jres.evec[:, 0]
beta      = -cv[1] / cv[0]
intercept = float((np.log(data.s[0, :q1]) - beta * np.log(data.s[1, :q1])).mean())
spread_q1 = np.log(data.s[0, :q1]) - beta * np.log(data.s[1, :q1]) - intercept
fixed_std = float(spread_q1.std())
phi       = AutoReg(spread_q1, lags=1).fit().params[1]
wn_var    = fixed_std ** 2 * (1 - phi ** 2)

print(f"\u03b2={beta:.4f}  intercept={intercept:.4f}  std={fixed_std:.4f}  \u03c6={phi:.4f}")
print(f"Training data (Q1-Q3): 0\u2013{q4}  ({q4} steps)")
print(f"Real test (Q4): {q4}\u2013{n}  ({n - q4} steps)")

N, T = 50, q4
log_s1 = np.cumsum(np.c_[np.full(N, np.log(100.0)),
                          0.0005 + 0.015 * np.random.randn(N, T - 1)], axis=1)
noise      = np.sqrt(wn_var) * np.random.randn(N, T)
spread_all = np.zeros((N, T))
for t in range(1, T):
    spread_all[:, t] = phi * spread_all[:, t - 1] + noise[:, t]

log_s0 = beta * log_s1 + intercept + spread_all
pairs  = np.stack([np.exp(log_s0), np.exp(log_s1)], axis=1)

print(f"Generated {N} synthetic series of length {T}  |  spread std: {spread_all.std(axis=1).mean():.4f} (target {fixed_std:.4f})")

In [ ]:
# ── LSTM to predict next-step spread ──────────────────────────────────────────
LOOKBACK = 20   # past steps fed into LSTM
HORIZON  = 1    # steps ahead to predict

class SpreadLSTM(nn.Module):
    def __init__(self, hidden=64, n_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, n_layers, batch_first=True)
        self.head = nn.Sequential(nn.ReLU(), nn.Linear(hidden, HORIZON))

    def forward(self, x):          # x: (B, T, 1)
        out, _ = self.lstm(x)
        return self.head(out[:, -1])  # (B, HORIZON)


def make_sequences(series, lookback, horizon):
    X, y = [], []
    for t in range(lookback, len(series) - horizon):
        X.append(series[t - lookback:t])
        y.append(series[t:t + horizon])
    return np.array(X, np.float32), np.array(y, np.float32)


Xs, ys = [], []
# All 50 synthetic spread series (normalised)
for i in range(N):
    Xi, yi = make_sequences(spread_all[i] / fixed_std, LOOKBACK, HORIZON)
    Xs.append(Xi); ys.append(yi)

# Real Q1-Q3 spread
real_spread_train = np.log(data.s[0, :q4]) - beta * np.log(data.s[1, :q4]) - intercept
Xr, yr = make_sequences(real_spread_train / fixed_std, LOOKBACK, HORIZON)
Xs.append(Xr); ys.append(yr)

X_all = np.concatenate(Xs)
y_all = np.concatenate(ys)
perm  = np.random.permutation(len(X_all))
X_all, y_all = X_all[perm], y_all[perm]
split = int(0.9 * len(X_all))

Xt = torch.from_numpy(X_all[:split, :, None])
yt = torch.from_numpy(y_all[:split])
Xv = torch.from_numpy(X_all[split:, :, None])
yv = torch.from_numpy(y_all[split:])

rnn_model = SpreadLSTM(hidden=64)
opt       = torch.optim.Adam(rnn_model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
BATCH, EPOCHS = 256, 50

print("Training spread LSTM...")
for epoch in range(EPOCHS):
    rnn_model.train()
    idx   = torch.randperm(len(Xt))
    tloss = 0.0; nb = 0
    for i in range(0, len(Xt), BATCH):
        b = idx[i:i + BATCH]
        loss = criterion(rnn_model(Xt[b]), yt[b])
        opt.zero_grad(); loss.backward(); opt.step()
        tloss += loss.item(); nb += 1
    if (epoch + 1) % 10 == 0:
        rnn_model.eval()
        with torch.no_grad():
            vloss = criterion(rnn_model(Xv), yv).item()
        print(f"  epoch {epoch+1:3d}  train={tloss/nb:.5f}  val={vloss:.5f}")

rnn_model.eval()
print("LSTM training complete.")

In [ ]:
# Quick visual: LSTM 1-step ahead prediction on Q4 (out-of-sample)
%matplotlib widget
import matplotlib.pyplot as plt

real_spread_q4 = np.log(data.s[0, q4:]) - beta * np.log(data.s[1, q4:]) - intercept

# Pre-fill buffer with last LOOKBACK steps of training spread
buf = ((np.log(data.s[0, q4 - LOOKBACK:q4])
        - beta * np.log(data.s[1, q4 - LOOKBACK:q4])
        - intercept) / fixed_std).astype(np.float32)

preds = []
for t in range(len(real_spread_q4) - 1):
    with torch.no_grad():
        p = rnn_model(torch.from_numpy(buf[None, :, None])).numpy()[0, 0] * fixed_std
    preds.append(p)
    buf = np.roll(buf, -1)
    buf[-1] = real_spread_q4[t] / fixed_std

preds = np.array(preds)
corr  = float(np.corrcoef(real_spread_q4[1:], preds)[0, 1])
print(f"1-step ahead prediction correlation (Q4): {corr:.4f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(real_spread_q4[1:], label="actual Q4 spread", alpha=0.8)
ax.plot(preds, label="LSTM 1-step prediction", alpha=0.7)
ax.set_title(f"LSTM spread prediction vs actual (Q4 out-of-sample)  |  corr={corr:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN
from env import Market


class SyntheticMarket(gym.Wrapper):
    """On each reset, samples a new random synthetic series."""
    def __init__(self, pairs, **market_kwargs):
        self._pairs = pairs
        env = Market(self._make_pair(0), **market_kwargs)
        super().__init__(env)

    def _make_pair(self, idx):
        pd = object.__new__(PairData)
        pd.s = self._pairs[idx]
        pd.labels = ["s0", "s1"]
        pd.timestamps = np.arange(self._pairs.shape[2])
        return pd

    def reset(self, **kwargs):
        idx = np.random.randint(len(self._pairs))
        self.env._pair_data = self._make_pair(idx)
        return self.env.reset(**kwargs)


class FlatRNNMarket(gym.Wrapper):
    """Flattens dict obs (10 features) and appends 2 frozen-LSTM features:
       [rnn_pred_z, rnn_pred_change_z]  →  12-dim obs.
    """
    OBS_DIM = 12

    def __init__(self, env, rnn_model, fixed_std, lookback):
        super().__init__(env)
        self._rnn = rnn_model
        self._std = fixed_std
        self._lb  = lookback
        self._buf = np.zeros(lookback, dtype=np.float32)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(self.OBS_DIM,), dtype=np.float32
        )

    def _update_and_predict(self, spread_val):
        self._buf = np.roll(self._buf, -1)
        self._buf[-1] = spread_val / self._std
        with torch.no_grad():
            x = torch.from_numpy(self._buf[None, :, None])
            pred = self._rnn(x).numpy()[0, 0] * self._std
        return pred

    def _flat(self, obs, pred):
        current = float(obs["spread"][0])
        return np.array([
            float(obs["position"]),
            abs(float(obs["spread_z_score"][0])),
            abs(float(obs["entry_z_score"][0])),
            abs(current),
            abs(float(obs["entry_spread"][0])),
            float(obs["macd"][0]),
            float(obs["macd_signal"][0]),
            float(obs["macd_hist"][0]),
            float(obs["rolling_mean"][0]),
            float(obs["rolling_std"][0]),
            pred / self._std,                    # RNN predicted next z-score
            (pred - current) / self._std,        # RNN predicted spread change (z-scaled)
        ], dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._buf = np.zeros(self._lb, dtype=np.float32)
        pred = self._update_and_predict(float(obs["spread"][0]))
        return self._flat(obs, pred), info

    def step(self, action):
        obs, reward, term, trunc, info = self.env.step(action)
        pred = self._update_and_predict(float(obs["spread"][0]))
        return self._flat(obs, pred), reward, term, trunc, info


train_env = FlatRNNMarket(
    SyntheticMarket(
        pairs,
        fixed_beta=beta,
        spread_intercept=intercept,
        fixed_std=fixed_std,
        reward_alpha=0.01,
        transaction_cost=0.004,
        squared_reward=False,
    ),
    rnn_model=rnn_model,
    fixed_std=fixed_std,
    lookback=LOOKBACK,
)

TOTAL_STEPS = N * T * 5
print(f"Training DQN+RNN for {TOTAL_STEPS:,} steps...")

model_rnn = DQN(
    "MlpPolicy", train_env, verbose=1,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=2000,
    batch_size=64,
    gamma=0.999,
    exploration_fraction=0.6,
    exploration_final_eps=0.1,
    target_update_interval=500,
    train_freq=4,
)
model_rnn.learn(total_timesteps=TOTAL_STEPS)
print("Done.")

In [ ]:
from static_thresh_base import ZScoreBaseline
from runner import run_strategy, plot_strategy


class SB3RNNAgent:
    """Eval agent: maintains its own LSTM buffer and produces augmented obs."""
    def __init__(self, model, rnn_model, fixed_std, lookback):
        self.model = model
        self._rnn  = rnn_model
        self._std  = fixed_std
        self._lb   = lookback
        self._buf  = np.zeros(lookback, dtype=np.float32)

    def reset(self):
        self._buf = np.zeros(self._lb, dtype=np.float32)

    def predict(self, obs):
        spread = float(obs["spread"][0])
        self._buf = np.roll(self._buf, -1)
        self._buf[-1] = spread / self._std
        with torch.no_grad():
            x    = torch.from_numpy(self._buf[None, :, None])
            pred = self._rnn(x).numpy()[0, 0] * self._std
        flat = np.array([
            float(obs["position"]),
            abs(float(obs["spread_z_score"][0])),
            abs(float(obs["entry_z_score"][0])),
            abs(spread),
            abs(float(obs["entry_spread"][0])),
            float(obs["macd"][0]),
            float(obs["macd_signal"][0]),
            float(obs["macd_hist"][0]),
            float(obs["rolling_mean"][0]),
            float(obs["rolling_std"][0]),
            pred / self._std,
            (pred - spread) / self._std,
        ], dtype=np.float32)
        action, _ = self.model.predict(flat, deterministic=True)
        return int(action)


def metrics(results, label, rf_annual=0.05):
    pnl            = results["pnl"]
    n_days         = len(pnl)
    daily_returns  = np.diff(pnl) / pnl[:-1]
    excess_returns = daily_returns - rf_annual / 252
    sharpe         = excess_returns.mean() / (excess_returns.std() + 1e-12) * np.sqrt(252)
    running_max    = np.maximum.accumulate(pnl)
    max_drawdown   = ((pnl - running_max) / running_max).min()
    total_return   = (pnl[-1] - pnl[0]) / pnl[0]
    ann_return     = (1 + total_return) ** (252 / n_days) - 1
    print(f"{label}")
    print(f"  Sharpe ratio      : {sharpe:.3f}")
    print(f"  Max drawdown      : {max_drawdown*100:.2f}%")
    print(f"  Annualised return : {ann_return*100:.2f}%")
    print()


env_kwargs = dict(fixed_beta=beta, spread_intercept=intercept, fixed_std=fixed_std)
q4_data = data.slice(q4)

agent_rnn = SB3RNNAgent(model_rnn, rnn_model, fixed_std, LOOKBACK)
results_rnn_q4 = run_strategy(Market(q4_data, **env_kwargs), agent_rnn)
print("--- DQN+RNN (real Q4 test) ---")
plot_strategy(results_rnn_q4)
metrics(results_rnn_q4, "DQN+RNN \u2014 real Q4")

results_base_q4 = run_strategy(
    Market(q4_data, **env_kwargs),
    ZScoreBaseline(entry_threshold=2.0, exit_threshold=0.5),
)
print("--- Z-score baseline (real Q4 test) ---")
plot_strategy(results_base_q4)
metrics(results_base_q4, "Z-score baseline \u2014 real Q4")

In [ ]:
# DQN+RNN on a single synthetic series (series 0)
synth_pair = object.__new__(PairData)
synth_pair.s = pairs[0]
synth_pair.labels = ["s0 (synth)", "s1 (synth)"]
synth_pair.timestamps = np.arange(pairs.shape[2])

agent_rnn_s = SB3RNNAgent(model_rnn, rnn_model, fixed_std, LOOKBACK)
results_rnn_synth = run_strategy(Market(synth_pair, **env_kwargs), agent_rnn_s)
print("--- DQN+RNN on synthetic series 0 ---")
plot_strategy(results_rnn_synth)
metrics(results_rnn_synth, "DQN+RNN \u2014 synthetic series 0")

results_base_synth = run_strategy(
    Market(synth_pair, **env_kwargs),
    ZScoreBaseline(entry_threshold=2.0, exit_threshold=0.5),
)
print("--- Z-score baseline on synthetic series 0 ---")
plot_strategy(results_base_synth)
metrics(results_base_synth, "Z-score baseline \u2014 synthetic series 0")

In [ ]:
# Unseen synthetic series (different seed, never seen during training)
rng_test = np.random.default_rng(999)
log_s1_u = np.cumsum(np.c_[np.full(1, np.log(100.0)),
                 0.0005 + 0.015 * rng_test.standard_normal((1, T - 1))], axis=1)
noise_u  = np.sqrt(wn_var) * rng_test.standard_normal((1, T))
spr_u    = np.zeros((1, T))
for t in range(1, T):
    spr_u[:, t] = phi * spr_u[:, t - 1] + noise_u[:, t]
log_s0_u = beta * log_s1_u + intercept + spr_u
pairs_u  = np.stack([np.exp(log_s0_u), np.exp(log_s1_u)], axis=1)

unseen = object.__new__(PairData)
unseen.s = pairs_u[0]
unseen.labels = ["s0 (unseen)", "s1 (unseen)"]
unseen.timestamps = np.arange(pairs_u.shape[2])

agent_rnn_u = SB3RNNAgent(model_rnn, rnn_model, fixed_std, LOOKBACK)
results_rnn_u = run_strategy(Market(unseen, **env_kwargs), agent_rnn_u)
print("--- DQN+RNN on unseen synthetic series ---")
plot_strategy(results_rnn_u)
metrics(results_rnn_u, "DQN+RNN \u2014 unseen synthetic")